In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
import os
import sys

import h5py
import healpy as hp
import numpy as np

sys.path.append(os.path.join(os.getcwd(), ".."))
from scripts.utils import remove_mono_dipole, setup_logging, plot_predictions

from ksw_joblib import KSW_joblib

from scripts import (
    core,
    Core,
)  # note the small c to import the core module and not the class

# Monkey patch the core module
core.KSW = KSW_joblib

In [ ]:
logger = setup_logging(__name__, level=logging.DEBUG)

## Heidelberg

In [ ]:
core = Core(["settings/heidelberg.json", "--nsims", "1", "--beam_width", "0"])

In [ ]:
def alm_step_loader(idx):
    # need this here instead of import
    logger.debug("Sending alm step %s", idx)
    return core.data.compute_alm_sim(core.lensing)


thetas = min(512, int(np.floor(1.5 * core.lmax + 1)))

mc_path = os.path.join(core.alm_dir, "kswmc")
os.makedirs(mc_path, exist_ok=True)

mc_file = os.path.join(mc_path, f"{core.base_name}.hdf5")
if os.path.exists(mc_file):
    os.remove(mc_file)
if os.path.exists(mc_file):
    logger.info("Loading KSW state from %s", mc_file)
    core.ksw.start_from_read_state(mc_file, None)
else:
    core.ksw.step_batch(alm_step_loader, range(100), theta_batch=thetas)

    logger.info("Saving KSW state to %s", mc_file)
    core.ksw.write_state(mc_file)

In [ ]:
fisher = float(core.ksw.compute_fisher())
fisher, np.sqrt(1 / fisher)

In [ ]:
fnls = np.random.uniform(core.fnl_min, core.fnl_max, 10)
idxs = range(1, 10)


def alm_hei_loader(idx):
    str_idx = str(idx).zfill(4)
    base1 = f"data/heidelberg/alm_l_{str_idx}_v3.fits"
    base2 = f"data/heidelberg/alm_nl_{str_idx}_v3.fits"

    alm_heidelberg_l = np.array(hp.read_alm(base1, hdu=1))
    alm_heidelberg_nl = np.array(hp.read_alm(base2, hdu=1))
    fnl = fnls[idx]

    alm = alm_heidelberg_l + fnl * alm_heidelberg_nl
    # alm *= 2.7255 * 10 ** (6)  # convert heidelberg to uK
    alm = remove_mono_dipole(alm)

    logger.info("sending idx: %s, fnl: %s" % (idx, fnl))
    return alm

hei_estimates = core.ksw.compute_estimate_batch(alm_hei_loader, idxs, fisher=fisher)

In [ ]:
plot_dir = os.path.join(core.plot_dir, "notebooks")
os.makedirs(plot_dir, exist_ok=True)
pred_file = os.path.join(plot_dir, f"{core.base_name}_preds.png")
plot_predictions(
    fnls[idxs],
    hei_estimates,
    fisher,
    # save_file=pred_file,
)

## Sim

In [ ]:
core = Core(["settings/planck.json"])

In [ ]:
mc_path = os.path.join(core.alm_dir, "kswmc")
os.makedirs(mc_path, exist_ok=True)
mc_file = os.path.join(mc_path, f"{core.base_name}.hdf5")
if os.path.exists(mc_file):
    logger.info("Loading KSW state from %s", mc_file)
    core.ksw.start_from_read_state(mc_file, None)
else:
    logger.info("Running KSW step")
    core.ksw.step_batch(alm_step_loader, range(100), theta_batch=thetas)
    ksw_ran = True  # dont need to do this later

    logger.info("Saving KSW state to %s", mc_file)
    core.ksw.write_state(mc_file)

In [ ]:
fisher = float(core.ksw.compute_fisher())
fisher, np.sqrt(1 / fisher)

In [ ]:
alm_file = h5py.File(core.alm_file, "r", swmr=True, locking=False)
alms = alm_file["alm"]
fnls = alm_file["fnl"]

def estimator_loader(idx):
    """Loads in a single alm given an idx."""
    sim, pol = np.unravel_index(int(idx), (core.total_sims, core.npol))
    logger.debug("Sending alm (%s, %s) with fnl %s", sim, pol, fnls[sim])
    return np.array(alms[sim, pol])

idxs = range(50)

sim_estimates = core.ksw.compute_estimate_batch(estimator_loader, idxs, fisher=fisher)

In [ ]:
idx, pol = np.unravel_index(idxs, (core.nsims, core.npol))
plot_predictions(fnls[idx], sim_estimates, fisher=fisher)